# Day 2 · Occupancy Exploration
**Owner:** Sloane · Issue #20

Builds on the Day 1 load (issue #11) and completes the Day 2 checklist for the
Occupancy dataset:

1. Nulls / duplicates / ranges check
2. Occupancy pattern analysis
3. Validation of the shared `sparkcityx.data_quality` module (Hakeem, issue #22)
   against Occupancy

Run `uv run python scripts/generate-data.py --records 36000` first so
`data/raw/occupancy_data.csv` exists locally before running this notebook.

## Setup

In [ ]:
import json
import os

os.environ.setdefault(
    "JAVA_HOME",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
)

from pyspark.sql import SparkSession, functions as F

from sparkcityx.data_quality import validate_dataframe

spark = (
    SparkSession.builder
    .appName("day2-occupancy-exploration")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Load Occupancy Data
(Confirms issue #11: CSV loading, schema check, row count.)

In [1]:
occupancy_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/raw/occupancy_data.csv")
)

print(f"Row count: {occupancy_df.count()}")
occupancy_df.printSchema()
occupancy_df.show(5, truncate=False)

Row count: 36000
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- available_rooms: integer (nullable = true)
 |-- occupied_rooms: integer (nullable = true)
 |-- guests: integer (nullable = true)

+---------+-------------------+------------+------------+---------------+--------------+------+
|sensor_id|timestamp          |location_lat|location_lon|available_rooms|occupied_rooms|guests|
+---------+-------------------+------------+------------+---------------+--------------+------+
|OCC-0001 |2025-01-01 00:00:00|40.680661   |-74.049734  |235            |130           |485   |
|OCC-0002 |2025-01-01 00:15:00|40.680237   |-74.046201  |28             |10            |19    |
|OCC-0003 |2025-01-01 00:30:00|40.680661   |-74.041989  |1227           |606           |831   |
|OCC-0004 |2025-01-01 00:45:00|40.680455   |-74.038697  |153            |86            |270   

## Data Quality Validation

Runs the shared `validate_dataframe` function (Hakeem, issue #22) against the real
Occupancy dataset — this both performs the nulls/duplicates/ranges check required by
issue #20 and validates the shared module for issue #20's last checklist item.

In [2]:
report = validate_dataframe(occupancy_df, "occupancy")
numeric_summary = report.pop("numeric_summary")
print(json.dumps(report, indent=2, default=str))

{
  "dataset_type": "occupancy",
  "valid": true,
  "record_count": 36000,
  "missing_columns": [],
  "non_numeric_columns": [],
  "null_counts": {},
  "duplicate_count": 0,
  "range_violations": {},
  "value_violations": {},
  "timestamp_violations": {},
  "cross_field_violations": {}
}


### Findings — Data Quality

- All 36,000 rows are **valid**: no missing columns, no nulls, no duplicate
  `(sensor_id, timestamp)` pairs, no range violations, and the
  `occupied_rooms <= available_rooms` cross-field rule holds for every row.
- The generated data is clean by construction, so the missing-data
  interpolation and outlier-treatment TODOs in `day2_stuff.txt` have nothing
  to act on for Occupancy — noted here rather than adding dead code that
  never triggers.

## Occupancy Pattern Analysis

In [3]:
patterns_df = (
    occupancy_df
    .withColumn("occ_rate", F.col("occupied_rooms") / F.col("available_rooms"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("is_weekend", F.dayofweek("timestamp").isin([1, 7]))
)

print("=== Avg occupancy rate by hour of day ===")
patterns_df.groupBy("hour").agg(
    F.round(F.avg("occ_rate"), 3).alias("avg_occ_rate"),
    F.round(F.avg("guests"), 1).alias("avg_guests"),
    F.count("*").alias("n"),
).orderBy("hour").show(24)

=== Avg occupancy rate by hour of day ===
+----+------------+----------+----+
|hour|avg_occ_rate|avg_guests|   n|
+----+------------+----------+----+
|   0|        0.71|     531.8|1500|
|   1|       0.712|     483.0|1500|
|   2|       0.708|     844.9|1496|
|   3|       0.709|     523.4|1504|
|   4|       0.707|     499.1|1500|
|   5|       0.718|     870.9|1500|
|   6|       0.713|     533.3|1500|
|   7|       0.707|     477.4|1500|
|   8|       0.703|     853.4|1500|
|   9|       0.714|     544.4|1500|
|  10|       0.709|     470.1|1500|
|  11|       0.711|     876.3|1500|
|  12|       0.708|     535.5|1500|
|  13|       0.702|     489.0|1500|
|  14|       0.709|     832.5|1500|
|  15|       0.704|     534.1|1500|
|  16|       0.712|     478.5|1500|
|  17|       0.711|     848.6|1500|
|  18|       0.709|     536.0|1500|
|  19|       0.709|     471.8|1500|
|  20|       0.713|     867.3|1500|
|  21|       0.709|     525.5|1500|
|  22|       0.703|     470.9|1500|
|  23|       0.719|   

In [4]:
print("=== Avg occupancy rate by day of week (1=Sun..7=Sat) ===")
patterns_df.groupBy("day_of_week").agg(
    F.round(F.avg("occ_rate"), 3).alias("avg_occ_rate"),
    F.round(F.avg("guests"), 1).alias("avg_guests"),
    F.count("*").alias("n"),
).orderBy("day_of_week").show()

=== Avg occupancy rate by day of week (1=Sun..7=Sat) ===
+-----------+------------+----------+----+
|day_of_week|avg_occ_rate|avg_guests|   n|
+-----------+------------+----------+----+
|          1|       0.769|     683.9|5088|
|          2|       0.691|     603.9|5088|
|          3|       0.687|     595.1|5088|
|          4|       0.681|     602.1|5184|
|          5|       0.683|     591.3|5184|
|          6|       0.686|     608.1|5184|
|          7|        0.77|     680.8|5184|
+-----------+------------+----------+----+


In [5]:
print("=== Weekend vs weekday ===")
patterns_df.groupBy("is_weekend").agg(
    F.round(F.avg("occ_rate"), 3).alias("avg_occ_rate"),
    F.round(F.avg("guests"), 1).alias("avg_guests"),
    F.count("*").alias("n"),
).show()

=== Weekend vs weekday ===
+----------+------------+----------+-----+
|is_weekend|avg_occ_rate|avg_guests|    n|
+----------+------------+----------+-----+
|      true|       0.769|     682.3|10272|
|     false|       0.686|     600.1|25728|
+----------+------------+----------+-----+


### Findings — Pattern Analysis

- **No strong intraday cycle**: average occupancy rate stays in a tight
  ~0.70–0.72 band across every hour of the day.
- **Clear weekly pattern**: weekend occupancy (Sat/Sun) averages ~0.77 vs.
  ~0.68 on weekdays — about a 12% relative lift, consistent with
  leisure-driven weekend demand.
- This weekly seasonality is worth carrying into the Day 3 cross-check
  (issue #33) against Financial data, since revenue is expected to follow
  the same seasonal pattern as occupancy.

In [6]:
spark.stop()